In [1]:
# notebooks/09_predict_test.ipynb
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
from src.db import DATA_RAW, MODELS_DIR
from src.predict import CarPricePredictor

# берём 5 случайных машин из датасета
df = pd.read_parquet(DATA_RAW / "cars.parquet")
sample = df.sample(5, random_state=1)

predictor = CarPricePredictor(MODELS_DIR)

for i, (_, row) in enumerate(sample.iterrows()):
    car = row.to_dict()                          # ← сырая карточка
    car.pop("price", None)                       # таргет не показываем
    car.pop("seller_description", None)

    result = predictor.predict(car)
    true_price = row["price"]

    err_pct = abs(result["price"] - true_price) / true_price * 100
    covered = result["range_80"][0] <= true_price <= result["range_80"][1]

    print(f"\n#{i}: {row['brand']} {row['model']} {row['year']}")
    print(f"  True     : {true_price:>12,.0f}")
    print(f"  Predicted: {result['price']:>12,.0f}  (err {err_pct:.1f}%)")
    print(f"  Range 80%: [{result['range_80'][0]:>12,.0f}, "
          f"{result['range_80'][1]:>12,.0f}]   "
          f"{'✓ covered' if covered else '✗ missed'}")


#0: Mercedes-Benz CLS-Class 2012
  True     :    2,349,077
  Predicted:    2,206,580  (err 6.1%)
  Range 80%: [   1,565,293,    2,802,218]   ✓ covered

#1: OMODA C5 2026
  True     :    2,779,000
  Predicted:    2,454,733  (err 11.7%)
  Range 80%: [   2,162,071,    2,758,693]   ✗ missed

#2: Renault Duster 2013
  True     :    1,100,000
  Predicted:      989,289  (err 10.1%)
  Range 80%: [     670,704,    1,265,319]   ✓ covered

#3: Lamborghini Urus 2022
  True     :   24,700,000
  Predicted:   26,018,542  (err 5.3%)
  Range 80%: [  12,463,667,   32,192,658]   ✓ covered

#4: Nissan X-Trail 2021
  True     :    2,950,000
  Predicted:    2,682,554  (err 9.1%)
  Range 80%: [   1,930,662,    3,068,239]   ✓ covered
